In [1]:
import numpy as np
from filterpy.kalman import ExtendedKalmanFilter
import jax

In [ ]:
'''
mu is the expected value of the flux of a given star
d id the desired transit depth --> koi_depth†, Transit Depth (parts per million)
k is the frequency
a is orbital distance
'''

def h(theta, R_star, a, mu, d):
    del_theta = 2*R_star / a
    k = (2/del_theta) * np.tan(0.99*(np.pi/2)) 
    return mu * (1 - d + 2*(d/np.pi))*np.abs(1/np.tan(k*np.sin(theta)))

def h_der(f):
    return jax.grad(f)

In [ ]:
for planet in data:

    #theta
    G = 6.67e-11
    dt = 0.01
    
    #fix these to be the measurements (I have no idea how to find this from the data)
    m = planet["Stellar Mass (solar mass)"]
    r = planet[radius]

    dim_x = 1
    dim_z = 1

    ekf = ExtendedKalmanFilter(dim_x, dim_z)

    #initial state
    #remember x is just theta
    ekf.x = np.array([[0.0]])
    ekf.P = np.array([[1.0]])

    #Linear State
    ekf.F = np.array([[1.0]])
    ekf.B = np.array([[1.0]])
    U = np.array([dt * np.sqrt((G * m)/r**3)])

    #Noise
    ekf.Q = np.array([[1e-10]])
    ekf.R = np.array([[20/10**6]])
    #ekf.R = np.array([[x*20/10**6]])

    for measurement in measurements:
        ekf.predict(u=U)
        ekf.update(z=measurement, HJacobian=h_der, Hx=h, hx_args=(R_star, a, mu, d))

In [ ]:
#plot the observed measurements and then our thetas turned into brightness via H